# LanceDB Tutorial: Getting Started with Vector Databases

This tutorial covers the essential LanceDB concepts you'll need for the benchmark retrieval notebook. LanceDB is a powerful vector database that handles embeddings automatically and provides multiple search methods in a single API.

## What You'll Learn

1. **Database Setup and Connection** - Creating and connecting to a LanceDB instance
2. **Schema Definition with Pydantic** - Using LanceModel to define table schemas
3. **Automatic Embedding Generation** - How LanceDB handles embeddings behind the scenes
4. **Data Ingestion** - Loading and structuring your data
5. **Search Methods** - Vector, full-text, and hybrid search
6. **Reranking** - Improving search results with rerankers

Let's dive in!

## 1. Database Setup and Connection

LanceDB can run locally or in the cloud. For development, we'll use a local instance that stores data in a directory.

In [1]:
import lancedb

In [2]:
db = lancedb.connect("./tutorial_db")

★ Insight ─────────────────────────────────────
  URI (Uniform Resource Identifier) is the broader concept that includes URLs. While all URLs are URIs, not all URIs are URLs. URIs identify resources, while URLs
  specifically locate them on a network.
  ─────────────────────────────────────────────────

  URI vs URL: The Key Differences

  URI (Uniform Resource Identifier)
  - Purpose: Identifies a resource (like giving something a unique name)
  - Scope: Can identify resources anywhere - locally, on networks, or abstractly
  - Examples:
    - file:///Users/you/data/database (local file path)
    - urn:isbn:1234567890 (book identifier, not a location)
    - mailto:user@example.com (email address)
    - https://example.com/page (this is also a URL)

  URL (Uniform Resource Locator)
  - Purpose: Locates a resource (tells you where to find it on a network)
  - Scope: Specifically for network-accessible resources
  - Examples:
    - https://www.google.com
    - ftp://files.example.com/document.pdf
    - http://localhost:8000/api/data

 Simple Memory Aid

  - URI = "Universal ID" (identifies anything)
  - URL = "Universal Location" (locates network resources)

  Think of it like addresses: A URI is like any kind of address (home address, email address, phone number), while a URL is specifically a street address you can
  drive to.

In [3]:
db.uri

'/Users/dipamvasani/Desktop/real_desktop/coding/systematically-improving-rag/latest/week1/tutorial_db'

In [4]:
db.table_names()

['tutorial_docs']

In [5]:
print(f"Connected to LanceDB at: {db.uri}")
print(f"Existing tables: {db.table_names()}")

Connected to LanceDB at: /Users/dipamvasani/Desktop/real_desktop/coding/systematically-improving-rag/latest/week1/tutorial_db
Existing tables: ['tutorial_docs']


## 2. Schema Definition with Pydantic

LanceDB uses Pydantic models to define table schemas. This is where the magic happens - you define which fields should be embedded automatically!

In [6]:
from lancedb.pydantic import LanceModel, Vector
from lancedb.embeddings import get_registry

#### LanceModel

- provides automatic embedding handling (vs manual if you use BaseModel)
- Adds vector database specific fields like Vector

In [7]:
# .get_registry() function returns LanceDB's embedding registry - a central catalog of all available embedding providers and models.
# .get("openai")  This retrieves the OpenAI provider from the registry. It's like asking: "Give me the OpenAI embedding service."
# .create This creates an embedding function instance configured for the specific model.

# Different providers have different APIs, but the registry normalizes them:

In [8]:
registry = get_registry()

In [9]:
# no way to list providers
[x for x in dir(registry) if not x.startswith("_")]

['function_to_metadata',
 'get',
 'get_instance',
 'get_table_metadata',
 'get_var',
 'parse_functions',
 'register',
 'reset',
 'set_var']

In [10]:
embedding_func = get_registry().get("openai").create(name="text-embedding-3-small")

In [11]:
print(f"Embedding function dimensions: {embedding_func.ndims()}")

Embedding function dimensions: 1536


In [12]:
emb = embedding_func.compute_query_embeddings("this is some text")

In [13]:
type(emb), len(emb), len(emb[0])

(list, 1, 1536)

In [14]:
class Document(LanceModel):
    id: str
    text: str = embedding_func.SourceField()
    vector: Vector(embedding_func.ndims()) = embedding_func.VectorField()
    category: str = "general"

In [15]:
print("Schema defined successfully!")
print(f"Text field will be automatically embedded into {embedding_func.ndims()}-dimensional vectors")

Schema defined successfully!
Text field will be automatically embedded into 1536-dimensional vectors


### Understanding the Schema

- `SourceField()`: Marks the field that should be embedded
- `VectorField()`: Where the embeddings will be stored (auto-generated)
- `Vector(dimensions)`: Specifies the vector dimensions

LanceDB will automatically generate embeddings for the `text` field and store them in the `vector` field.

## 3. Creating Tables and Ingesting Data

Now let's create a table and add some sample data to practice with.

In [16]:
# Sample data - mix of technical and general content
sample_data = [
    {
        "id": "doc_1", 
        "text": "Machine learning is a subset of artificial intelligence that focuses on algorithms that learn from data.",
        "category": "tech"
    },
    {
        "id": "doc_2", 
        "text": "Python is a popular programming language for data science and machine learning applications.",
        "category": "tech"
    },
    {
        "id": "doc_3", 
        "text": "The best pizza in New York can be found at small local joints, not the famous tourist spots.",
        "category": "food"
    },
    {
        "id": "doc_4", 
        "text": "Vector databases store high-dimensional vectors and enable fast similarity search.",
        "category": "tech"
    },
    {
        "id": "doc_5", 
        "text": "Cooking at home saves money and allows you to control the quality of ingredients.",
        "category": "food"
    }
]

In [17]:
table = db.create_table("tutorial_docs", schema=Document, mode="overwrite")

### ⏺ ★ Insight ─────────────────────────────────────
  LanceDB uses versioned storage - each operation creates a new version, enabling time-travel queries and rollbacks. The version number tracks your table's
  evolution as you add or modify data.
  ─────────────────────────────────────────────────

  The AddResult(version=2) means your data was successfully added to the table, and this operation created version 2 of your table.

  Version1 is the empty table that we had

In [18]:
table.add(sample_data)

AddResult(version=6)

In [19]:
df = table.to_pandas(); df

,id,text,vector,category
0,doc_1,Machine learning is a subset of artificial int...,"[-0.009257173, -0.06275806, 0.023426317, -0.01...",tech
1,doc_2,Python is a popular programming language for d...,"[0.024406027, -0.0229987, 0.022072826, -0.0067...",tech
2,doc_3,The best pizza in New York can be found at sma...,"[-0.03484092, -0.009728874, -0.019904427, 0.01...",food
3,doc_4,Vector databases store high-dimensional vector...,"[-0.058570247, 0.0153894005, 0.011310077, 0.00...",tech
4,doc_5,Cooking at home saves money and allows you to ...,"[0.0060601365, -0.023790397, -0.0022591874, 0....",food


In [20]:
df.iloc[0].to_dict()

{'id': 'doc_1',
 'text': 'Machine learning is a subset of artificial intelligence that focuses on algorithms that learn from data.',
 'vector': array([-0.00925717, -0.06275806,  0.02342632, ..., -0.01801955,
        -0.0026539 ,  0.03332223], dtype=float32),
 'category': 'tech'}

In [21]:
print(f"Created table with {table.count_rows()} documents")
print(f"Table schema: {table.schema}")

Created table with 5 documents
Table schema: id: string not null
text: string not null
vector: fixed_size_list<item: float>[1536]
  child 0, item: float
category: string not null
-- schema metadata --
embedding_functions: '[
  {
    "name": "openai",
    "model": {
      "n' + 102


## 4. Setting Up Full-Text Search

For hybrid search (combining vector and keyword search), we need to create a full-text search index.

In [22]:
# table.create_index? # vector index, or other indexes
# table.create_scalar_index? # for values like timestamps, category, user_id, etc

In [23]:
# Create full-text search index on the 'text' field
# replace=True recreates index if you add more data or change schema
table.create_fts_index("text", replace=True)

print("Full-text search index created successfully!")
print("Now we can perform vector, full-text, and hybrid searches")

Full-text search index created successfully!
Now we can perform vector, full-text, and hybrid searches


### ⏺ ★ Insight ─────────────────────────────────────
  FTS indexes use inverted index structures - mapping each unique word to all documents containing it. This enables instant keyword lookups without scanning every
  document. The replace=True parameter handles index updates by recreating rather than erroring on conflicts.
  ─────────────────────────────────────────────────

In [24]:
 # Word → Document IDs
 #  ┌─────────────┬─────────────────┐
 #  │ Word        │ Documents       │
 #  ├─────────────┼─────────────────┤
 #  │ "python"    │ [1, 2]         │
 #  │ "programming"│ [1, 3]         │
 #  │ "is"        │ [1]            │
 #  │ "fun"       │ [1, 3]         │
 #  │ "machine"   │ [2]            │
 #  │ "learning"  │ [2]            │
 #  │ "with"      │ [2]            │
 #  │ "projects"  │ [3]            │
 #  │ "in"        │ [3]            │
 #  └─────────────┴─────────────────┘

 # How FTS Search Works

  # Query: "python programming"
  # 1. Tokenize: ["python", "programming"]
  # 2. Lookup each word:
  #    - "python" → documents [1, 2]
  #    - "programming" → documents [1, 3]
  # 3. Combine results: intersection [1] or union [1, 2, 3]
  # 4. Return matching documents instantly!

## 5. Search Methods

LanceDB provides three main search methods:
1. **Vector Search** - Semantic similarity using embeddings
2. **Full-Text Search (FTS)** - Keyword matching
3. **Hybrid Search** - Combines both methods

Let's try each method with the same query to see the differences.

In [25]:
query = "What programming languages are good for AI?"

In [26]:
vec_results = table.search(query, query_type="vector").limit(3).to_list();

In [27]:
len(vec_results)

3

In [28]:
# not sure what distance here is?
vec_results[0]['text'], vec_results[0]['_distance']

('Python is a popular programming language for data science and machine learning applications.',
 1.035424828529358)

In [29]:
print(vec_results[1]['text'])
print(vec_results[2]['text'])

Machine learning is a subset of artificial intelligence that focuses on algorithms that learn from data.
Vector databases store high-dimensional vectors and enable fast similarity search.


In [30]:
fts_results = table.search(query, query_type="fts").limit(3).to_list()

In [31]:
len(fts_results)

1

In [32]:
# TF-IDF score
fts_results[0]['_score']

2.890850782394409

In [33]:
fts_results[0]['text']

'Python is a popular programming language for data science and machine learning applications.'

In [34]:
# How does hybrid combine the results? Using Reciprocal rank fusion

# Step 1: Run both searches separately
# vector_results = table.search(query, query_type="vector").limit(10)
# fts_results = table.search(query, query_type="fts").limit(10)

# Step 2: Rank each result set
# Vector: [doc_A(0.2), doc_B(0.4), doc_C(0.6), ...]  # by distance
# FTS:    [doc_B(5.2), doc_D(4.8), doc_A(4.1), ...]  # by relevance score

# Step 3: Apply Reciprocal Rank Fusion


hybrid_results = table.search(query, query_type="hybrid").limit(3).to_list()

In [35]:
len(hybrid_results)

3

In [36]:
hybrid_results[0]['text']

'Python is a popular programming language for data science and machine learning applications.'

In [37]:
print(hybrid_results[1]['text'])
print(hybrid_results[2]['text'])

Machine learning is a subset of artificial intelligence that focuses on algorithms that learn from data.
Vector databases store high-dimensional vectors and enable fast similarity search.


### Understanding Search Results

- **Vector Search**: Finds semantically similar content even without exact keyword matches
- **Full-Text Search**: Looks for exact word matches (may return no results if words don't match exactly)
- **Hybrid Search**: Combines both approaches for potentially better results

Let's try a more specific query to see FTS in action:

In [38]:
# Try a query with exact keywords from our data
specific_query = "Python programming language"

print(f"Specific Query: '{specific_query}'\n")

print("=== FTS with exact keywords ===")
fts_specific = table.search(specific_query, query_type="fts").limit(3).to_list()
for result in fts_specific:
    print(f"[{result['id']}] {result['text']}")

print("\n=== Vector search with same query ===")
vector_specific = table.search(specific_query, query_type="vector").limit(3).to_list()
for result in vector_specific:
    print(f"[{result['id']}] {result['text']}")

Specific Query: 'Python programming language'

=== FTS with exact keywords ===
[doc_2] Python is a popular programming language for data science and machine learning applications.

=== Vector search with same query ===
[doc_2] Python is a popular programming language for data science and machine learning applications.
[doc_1] Machine learning is a subset of artificial intelligence that focuses on algorithms that learn from data.
[doc_4] Vector databases store high-dimensional vectors and enable fast similarity search.


## 6. Reranking

Rerankers can improve search results by reordering them based on more sophisticated scoring. Let's see how to use Cohere's reranker (the same one used in the benchmark notebook).

In [39]:
from lancedb.rerankers import CohereReranker

In [40]:
from lancedb.rerankers import CohereReranker

reranker = CohereReranker(
    model_name="rerank-english-v3.0", 
    column="text"
)

print("Reranker created successfully!")

Reranker created successfully!


In [41]:
vec_results = table.search(query, query_type="vector").limit(5).rerank(reranker=reranker).to_list()

In [42]:
5/0

ZeroDivisionError: division by zero

In [ ]:
# TODO(human): Compare search results with and without reranking
# TODO(human): Use the query "machine learning algorithms" and show:
# TODO(human): 1. Vector search results (limit 5)
# TODO(human): 2. The same search with reranking applied
# TODO(human): Print both results to see how reranking changes the order
# TODO(human): Format: print(f"[{result['id']}] {result['text'][:80]}...") for each result

● **Learn by Doing**

**Context:** I've set up a LanceDB table with sample documents and a Cohere reranker. The reranker can improve search result ordering by using more sophisticated scoring than basic vector similarity. This is particularly useful when you want the most relevant results at the top.

**Your Task:** In the cell above, implement a comparison between regular vector search and reranked search. Look for TODO(human) comments. Use the query "machine learning algorithms" and show both the regular vector search results and the reranked results.

**Guidance:** Use `table.search(query, query_type="vector").limit(5)` for regular search, and add `.rerank(reranker=reranker)` before `.to_list()` for reranked results. The reranker analyzes the query-document relevance more deeply than simple vector similarity.

## 7. Advanced Patterns

Here are some patterns you'll see in the benchmark notebook and real applications:

In [ ]:
# Pattern 1: Check if table exists before creating
def get_or_create_table(db, table_name, schema_class, data=None):
    """Get existing table or create new one with data"""
    if table_name in db.table_names():
        print(f"Table '{table_name}' already exists")
        table = db.open_table(table_name)
    else:
        print(f"Creating new table '{table_name}'")
        table = db.create_table(table_name, schema=schema_class, mode="overwrite")
        if data:
            table.add(data)
    
    # Always recreate FTS index (in case it's missing)
    table.create_fts_index("text", replace=True)
    return table

# Example usage
tutorial_table = get_or_create_table(
    db, 
    "tutorial_example", 
    Document, 
    sample_data
)

print(f"Table has {tutorial_table.count_rows()} rows")

In [ ]:
# Pattern 2: Flexible search function (like in the benchmark notebook)
def flexible_search(
    table, 
    query, 
    max_k=10, 
    search_type="vector", 
    reranker=None
):
    """Flexible search function supporting different modes and reranking"""
    
    # Start the search
    results = table.search(query, query_type=search_type).limit(max_k)
    
    # Apply reranking if provided
    if reranker:
        results = results.rerank(reranker=reranker)
    
    # Convert to list and return structured data
    return [
        {
            "id": result["id"], 
            "text": result["text"],
            "category": result["category"]
        } 
        for result in results.to_list()
    ]

# Test the flexible search
test_results = flexible_search(
    tutorial_table, 
    "vector databases", 
    max_k=3, 
    search_type="vector"
)

print("Flexible search results:")
for result in test_results:
    print(f"- [{result['id']}] {result['text'][:50]}...")

## 8. Working with Different Embedding Models

In the benchmark notebook, you'll see tables created with different embedding models. Here's how that works:

In [ ]:
# Different embedding models have different dimensions
small_embedding = get_registry().get("openai").create(name="text-embedding-3-small")
large_embedding = get_registry().get("openai").create(name="text-embedding-3-large")

print(f"Small model dimensions: {small_embedding.ndims()}")
print(f"Large model dimensions: {large_embedding.ndims()}")

# You need separate schemas for different embedding models
class DocumentSmall(LanceModel):
    id: str
    text: str = small_embedding.SourceField()
    vector: Vector(small_embedding.ndims()) = small_embedding.VectorField()

class DocumentLarge(LanceModel):
    id: str
    text: str = large_embedding.SourceField()
    vector: Vector(large_embedding.ndims()) = large_embedding.VectorField()

print("\nSchemas created for both embedding models")
print("In practice, you'd create separate tables: 'docs_small' and 'docs_large'")

## 9. Preparing for Evaluation

The benchmark notebook evaluates different configurations. Here's how search results are typically structured for evaluation:

In [ ]:
# Pattern 3: Format results for evaluation
def search_for_evaluation(table, query, **search_params):
    """Search and format results for evaluation metrics"""
    
    results = flexible_search(table, query, **search_params)
    
    # Format for evaluation (id and text pairs)
    formatted_results = [
        {"id": result["id"], "text": result["text"]} 
        for result in results
    ]
    
    return formatted_results

# Example evaluation search
eval_results = search_for_evaluation(
    tutorial_table, 
    "machine learning",
    max_k=5,
    search_type="vector"
)

print("Results formatted for evaluation:")
for i, result in enumerate(eval_results, 1):
    print(f"{i}. ID: {result['id']} | Text: {result['text'][:40]}...")

## 10. Cleanup and Summary

Let's clean up and summarize what we've learned:

In [ ]:
# Show final table status
print("=== Tutorial Summary ===")
print(f"Database location: {db.uri}")
print(f"Tables created: {db.table_names()}")
print(f"Total documents in main table: {table.count_rows()}")

print("\n=== Key LanceDB Concepts Learned ===")
print("✓ Database connection with lancedb.connect()")
print("✓ Schema definition with LanceModel and embedding functions")
print("✓ Automatic embedding generation for specified fields")
print("✓ Data ingestion with table.add()")
print("✓ Full-text search index creation")
print("✓ Three search types: vector, fts, hybrid")
print("✓ Reranking with external services")
print("✓ Flexible search functions for evaluation")

print("\n=== Ready for the Benchmark Notebook! ===")
print("You now understand how LanceDB works in the retrieval benchmark.")

## Next Steps

Now that you understand LanceDB basics, you're ready to dive into the benchmark notebook! Here's what to expect:

1. **Larger Dataset**: The benchmark uses the "bird-rag" dataset with real SQL queries
2. **Multiple Tables**: Separate tables for different embedding models
3. **Systematic Evaluation**: Comparing recall and MRR metrics across configurations
4. **Real-world Patterns**: The same patterns you learned here, applied to actual evaluation

### Key Takeaways for the Benchmark Notebook:

- `get_or_create_lancedb_table()` follows the pattern you learned in section 7
- The `retrieve()` function is similar to your `flexible_search()` function
- Multiple embedding models require separate tables with different schemas
- Search results are formatted for evaluation metrics

You're all set to understand and modify the benchmark code with confidence!